# Week 5: SQL practice

Save a copy with `_mine` in its name. Run setup, fill the `____` blanks, then write your own queries. Use `sql("""...""")` to run SQL in a code cell.

## Setup · run once

SF311 requests from August 3–9, 2026. `requests`: one row per request; `districts`: one row per district. Setup excludes `Test` requests and keeps missing channels.

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
from IPython.display import display

COURSE_ROOT = None  # Set to your course folder only if automatic discovery fails.
candidates = [Path(COURSE_ROOT).expanduser()] if COURSE_ROOT else []
for parent in [Path.cwd(), *Path.cwd().parents]:
    candidates.extend([parent, parent / "compss-211a", parent / "COMPSS-211A"])
ROOT = next((p for p in candidates if (p / "data/sf311_requests.csv").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Set COURSE_ROOT to your complete course folder and rerun.")
DATA = ROOT / "data"

raw = pd.read_csv(DATA / "sf311_requests.csv", dtype={"request_id": "string", "district": "Int64"})
requests = raw.loc[(raw["channel"] != "Test") | raw["channel"].isna()].copy()
districts = pd.read_csv(DATA / "sf_supervisor_districts.csv")
districts = districts.rename(columns={"sup_dist": "district", "sup_dist_name": "district_name"})
districts = districts[["district", "district_name"]]
assert requests["request_id"].notna().all() and requests["request_id"].is_unique
assert districts["district"].notna().all() and districts["district"].is_unique

if "database" in globals():
    database.close()
database = sqlite3.connect(":memory:")
requests.to_sql("requests", database, index=False)
districts.to_sql("districts", database, index=False)

def sql(query):
    return pd.read_sql_query(query, database)

print("Ready:", len(requests), "requests and", len(districts), "districts")
display(sql("SELECT * FROM requests LIMIT 3"))
display(sql("SELECT * FROM districts LIMIT 3"))

## 1. Select and filter

Run the example. Then complete the second query to show five **Phone** requests: ID, category, and district.

In [ ]:
sql("""
    SELECT request_id, category, district
    FROM requests
    ORDER BY request_id
    LIMIT 5
""")

In [ ]:
sql("""
    SELECT request_id, category, district
    FROM requests
    WHERE channel = '____'
    ORDER BY request_id
    LIMIT 5
""")

## 2. Count groups

Run the channel example. Then complete the query: how many Phone requests came from each district? Show the largest count first.

In [ ]:
# Worked example: one row per reporting channel.
sql("""
    SELECT channel, COUNT(*) AS requests
    FROM requests
    GROUP BY channel
    ORDER BY requests DESC, channel ASC
""")

In [ ]:
sql("""
    SELECT ____, COUNT(*) AS requests
    FROM requests
    WHERE channel = 'Phone'
    GROUP BY ____
    ORDER BY requests ____, district ASC
""")

## 3. Find missing values

Run the counts example. Then complete the query to count requests with an unknown district. Hint: `IS NULL`.

In [ ]:
# Worked example: compare all rows with observed durations.
sql("""
    SELECT COUNT(*) AS requests,
           COUNT(hours_to_close) AS with_time,
           AVG(hours_to_close) AS mean_hours
    FROM requests
""")

In [ ]:
sql("""
    SELECT COUNT(*) AS requests
    FROM requests
    WHERE district ____
""")

## 4. Join two tables

Match requests to district names using `district`. Keep every request. Fill both blanks, then run the check.

In [ ]:
joined = sql("""
    SELECT r.request_id, r.category, r.district, d.district_name
    FROM requests AS r
    LEFT JOIN districts AS d
        ON r.____ = d.____
    ORDER BY r.request_id
""")
joined.head()

In [ ]:
print("Before:", len(requests), "After:", len(joined))
print("Unique request IDs:", joined["request_id"].nunique())
print("Unmatched requests:", joined["district_name"].isna().sum())
assert len(joined) == len(requests)
assert joined["request_id"].notna().all() and joined["request_id"].is_unique
assert set(joined["request_id"]) == set(requests["request_id"])

## 5. Write your own SQL

Start from an empty cell; earlier examples are available for reference.

1. Show the five categories with the most **Open** requests, with their counts.
2. Choose a channel. Count its requests by **district name**, keeping requests without a match. This needs both tables.

In [ ]:
# Write query 1 here.

In [ ]:
# Write query 2 here.

## 6. Food-access briefing · 20 minutes on your own

You are an analyst at a California food-access nonprofit. Prepare a preliminary county briefing using **real USDA and Census data**. Run the loader when asked.

- `tracts`: one row per census tract (a small statistical area). `area_type` is Urban/Rural; `population_2020` counts residents.
- `counties`: one row per county, with its code and name.
- `low_access = 1`: USDA flags limited driving access to SNAP-authorized retailers (1-mile urban / 10-mile rural threshold).
- `low_income_low_access = 1`: also meets USDA's low-income criteria. Flags are 1/0; blanks are missing. [Definitions and sources](food_access_sources.md).

1. Identify the keys and connection between tables.
2. Choose a question about urban/rural differences; answer with a filter and grouped count.
3. Count **low-income/low-access tracts by county name**. Keep every qualifying tract and check the joined total against the source total.
4. Write two briefing sentences: a finding with count and denominator, then a limitation and what you would investigate next.

Early finish: compare a county's flagged count with **all** its tracts. Does the denominator change your interpretation?

In [ ]:
# Supplied: load the new tables. Preserve geographic codes as text.
for table, filename in {
    "tracts": "week05_food_access_tracts.csv",
    "counties": "week05_ca_counties.csv",
}.items():
    frame = pd.read_csv(DATA / filename, dtype={"tract_id": "string", "county_fips": "string"})
    frame.to_sql(table, database, index=False, if_exists="replace")

display(sql("SELECT * FROM tracts LIMIT 5"))
display(sql("SELECT * FROM counties LIMIT 5"))

Keys and connection: _write here._

In [ ]:
# My question:
# Write your SQL here.

In [ ]:
# Count low-income/low-access tracts by county name.

In [ ]:
# Check that the join retains every qualifying tract, without duplication.

Briefing finding (count and denominator): _write here._

Limitation and next investigation: _write here._